# Введение в MapReduce модель на Python


In [1]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

In [2]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [3]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [4]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [5]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [6]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [7]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [8]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [9]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [10]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [11]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [12]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных.

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [13]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*

mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL

In [14]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication

In [15]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])

def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(2.671140012491868)),
 (1, np.float64(2.671140012491868)),
 (2, np.float64(2.671140012491868)),
 (3, np.float64(2.671140012491868)),
 (4, np.float64(2.671140012491868))]

## Inverted index

In [16]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)

def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)

def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('what', ['0', '1']),
 ('is', ['0', '1', '2']),
 ('it', ['0', '1', '2']),
 ('banana', ['2']),
 ('a', ['2'])]

## WordCount

In [17]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [18]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]

def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)

  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*

flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount

In [19]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)

  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

# try to set COMBINER=REDUCER and look at the number of values sent over the network
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6), ('banana', 2)]),
 (1, [('a', 2), ('is', 18), ('it', 18), ('what', 10)])]

## TeraSort

In [20]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for value in split:
        yield (value, None)

  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])

def MAP(value:int, _):
  yield (value, None)

def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, np.float64(0.006241829102966223)),
   (None, np.float64(0.023035187334819573)),
   (None, np.float64(0.07512014288246482)),
   (None, np.float64(0.0766005141334537)),
   (None, np.float64(0.13513586364764885)),
   (None, np.float64(0.2642864217317399)),
   (None, np.float64(0.3022857898799325)),
   (None, np.float64(0.3179135159491354)),
   (None, np.float64(0.39990554275122037)),
   (None, np.float64(0.4204157269127269)),
   (None, np.float64(0.42609329276880703)),
   (None, np.float64(0.4660000424929903)),
   (None, np.float64(0.469187111356149))]),
 (1,
  [(None, np.float64(0.5061860091644895)),
   (None, np.float64(0.513852203943687)),
   (None, np.float64(0.5150400389979547)),
   (None, np.float64(0.523428084708736)),
   (None, np.float64(0.5288933889887251)),
   (None, np.float64(0.5680281121602183)),
   (None, np.float64(0.5785466853830012)),
   (None, np.float64(0.586256668192117)),
   (None, np.float64(0.6724948687355932)),
   (None, np.float64(0.684916045299487

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [21]:
input_collection = [0, 4, 2, 5, 8, 6, 1, 10, 3, 7, 9]

def RECORDREADER():
    return [(i, val) for i, val in enumerate(input_collection)]

def MAP(_, value: int):
    yield ("max", value)

def REDUCE(key: str, values: Iterator[int]):
    result = max(values)
    yield (key, result)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)

print(output) # Должно быть 10

[('max', 10)]


### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [22]:
input_collection = [0, 4, 2, 5, 8, 6, 1, 10, 3, 7, 9]

def RECORDREADER():
    return [(i, val) for i, val in enumerate(input_collection)]

def MAP(_, value: int):
    yield ("average", value)

def REDUCE(key: str, values: Iterator[int]):
    sum = 0
    count = 0
    for value in values:
        sum += value
        count += 1
    if (count > 0):
        yield (key, sum/count)
    else:
        yield (key, 0)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)

print(output) # Должно быть 5

[('average', 5.0)]


### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [23]:
def groupbykey_sort(iterable):
    if not iterable:
        return []

    sorted_items = sorted(iterable, key=lambda x: x[0])
    
    result = []
    current_key, current_value = sorted_items[0]
    current_group = [current_value]
    
    for i in range(1, len(sorted_items)):
        key, value = sorted_items[i]
        if key == current_key:
            current_group.append(value)
        else:
            result.append((current_key, current_group))
            current_key = key
            current_group = [value]

    result.append((current_key, current_group))
    return result

def MapReduceWithSort(RECORDREADER, MAP, REDUCE):
    return flatten(map(lambda x: REDUCE(*x), groupbykey_sort(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

В следующих примерах используется модифицированная функция MapReduceWithSort, в которой используется GroupByKey на основе сортировки

In [24]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduceWithSort(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

In [25]:
def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])

def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])

output = MapReduceWithSort(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(2.671140012491868)),
 (1, np.float64(2.671140012491868)),
 (2, np.float64(2.671140012491868)),
 (3, np.float64(2.671140012491868)),
 (4, np.float64(2.671140012491868))]

In [26]:
d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)

def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)

def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduceWithSort(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('a', ['2']),
 ('banana', ['2']),
 ('is', ['0', '1', '2']),
 ('it', ['0', '1', '2']),
 ('what', ['0', '1'])]

In [27]:
d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduceWithSort(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('a', 1), ('banana', 1), ('is', 9), ('it', 9), ('what', 5)]

### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [28]:
input_collection = ["apple", "apple", "orange", "banana", "banana", "apple", "grape", "banana", "grape", "orange", "orange"]
maps = 3
reducers = 2

def INPUTFORMAT():
    global maps

    def RECORDREADER(split):
        for i, value in enumerate(split):
            yield (i, value)

    split_size = int(np.ceil(len(input_collection) / maps))
    for i in range(0, len(input_collection), split_size):
        yield RECORDREADER(input_collection[i:i + split_size])

def MAP(item_id: int, item_value: str):
    yield (item_value, None)

def COMBINER(item: str, values: Iterator[None]):
    yield (item, None)

def REDUCE(item: str, values: Iterator[None]):
    yield (item, None)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=COMBINER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
print(partitioned_output)

final_result = []

for partition_id, partition in partitioned_output:
    partition_items = list(partition)
    for key, val in partition_items:
        final_result.append(key)

print('\n', final_result)

8 key-value pairs were sent over a network.
[(0, [('banana', None), ('orange', None)]), (1, [('apple', None), ('grape', None)])]

 ['banana', 'orange', 'apple', 'grape']


#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [29]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def RECORDREADER():
    return [(u.id, u) for u in input_collection]

def MAP(_, row: NamedTuple):
    if row.age > 30:
        yield (row, row)

def REDUCE(row: NamedTuple, rows: Iterator[NamedTuple]):
    yield row

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output # Предикат - Age > 30

[User(id=0, age=55, social_contacts=20, gender='male'),
 User(id=3, age=33, social_contacts=800, gender='female')]

### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [30]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

class ProjectedUser(NamedTuple):
    age: int

def RECORDREADER():
    return [(u.id, u) for u in input_collection]

def MAP(_, row: User):
    t = ProjectedUser(age=row.age)
    yield (t, t)

def REDUCE(t_prime: ProjectedUser, occurrences: Iterator[ProjectedUser]):
    yield t_prime

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output # Проекция на возраст

[ProjectedUser(age=55), ProjectedUser(age=25), ProjectedUser(age=33)]

### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [31]:
R = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

S = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
]

input_collection = R + S

def RECORDREADER():
    return [(u.id, u) for u in input_collection]

def MAP(_, t: User):
    yield (t, t)

def REDUCE(t: User, values: Iterator[User]):
    yield t

print('R:')
for row in R:
    print(row)

print('S:')
for row in S:
    print(row)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
print('Результат')
output

R:
User(id=0, age=55, social_contacts=20, gender='male')
User(id=1, age=25, social_contacts=240, gender='female')
User(id=2, age=25, social_contacts=500, gender='female')
User(id=3, age=33, social_contacts=800, gender='female')
S:
User(id=0, age=55, social_contacts=20, gender='male')
User(id=1, age=25, social_contacts=240, gender='female')
Результат


[User(id=0, age=55, social_contacts=20, gender='male'),
 User(id=1, age=25, social_contacts=240, gender='female'),
 User(id=2, age=25, social_contacts=500, gender='female'),
 User(id=3, age=33, social_contacts=800, gender='female')]

### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [32]:
R = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

S = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
]

input_collection = R + S

def RECORDREADER():
    return [(u.id, u) for u in input_collection]

def MAP(_, t: User):
    yield (t, t)

def REDUCE(t: User, values: Iterator[User]):
    if (len(values) == 2):
        yield t

print('R:')
for row in R:
    print(row)

print('S:')
for row in S:
    print(row)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
print('Результат')
output

R:
User(id=0, age=55, social_contacts=20, gender='male')
User(id=1, age=25, social_contacts=240, gender='female')
User(id=2, age=25, social_contacts=500, gender='female')
User(id=3, age=33, social_contacts=800, gender='female')
S:
User(id=0, age=55, social_contacts=20, gender='male')
User(id=1, age=25, social_contacts=240, gender='female')
Результат


[User(id=0, age=55, social_contacts=20, gender='male'),
 User(id=1, age=25, social_contacts=240, gender='female')]

### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [33]:
R = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

S = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
]

def RECORDREADER():
    R_data = [(u, 'R') for u in R]
    S_data = [(u, 'S') for u in S]
    return R_data + S_data

def MAP(t: User, relation: str):
    yield (t, relation)

def REDUCE(t: User, relations: Iterator[str]):
    if (list(relations) == ['R']):
        yield t

print('R:')
for row in R:
    print(row)

print('S:')
for row in S:
    print(row)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
print('Результат')
output

R:
User(id=0, age=55, social_contacts=20, gender='male')
User(id=1, age=25, social_contacts=240, gender='female')
User(id=2, age=25, social_contacts=500, gender='female')
User(id=3, age=33, social_contacts=800, gender='female')
S:
User(id=0, age=55, social_contacts=20, gender='male')
User(id=1, age=25, social_contacts=240, gender='female')
Результат


[User(id=2, age=25, social_contacts=500, gender='female'),
 User(id=3, age=33, social_contacts=800, gender='female')]

### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [34]:
class RelationR(NamedTuple):
    a: str
    b: int

class RelationS(NamedTuple):
    b: int
    c: str

R = [
    RelationR(a="apple", b=1),
    RelationR(a="banana", b=1),
    RelationR(a="grape", b=2)
]

S = [
    RelationS(b=1, c="red"),
    RelationS(b=1, c="yellow"),
    RelationS(b=2, c="green"),
    RelationS(b=3, c="blue")
]

def RECORDREADER():
    R_data = [(r, 'R') for r in R]
    S_data = [(s, 'S') for s in S]
    return R_data + S_data

def MAP(record, relation_name: str):
    if relation_name == 'R':
        yield (record.b, ('R', record.a))
    elif relation_name == 'S':
        yield (record.b, ('S', record.c))

def REDUCE(b: int, values: Iterator[tuple]):
  
    list_R = []
    list_S = []

    for tag, value in values:
        if tag == 'R':
            list_R.append(value)
        elif tag == 'S':
            list_S.append(value)

    for a in list_R:
        for c in list_S:
            yield (a, b, c)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('apple', 1, 'red'),
 ('apple', 1, 'yellow'),
 ('banana', 1, 'red'),
 ('banana', 1, 'yellow'),
 ('grape', 2, 'green')]

### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [35]:
class CustomUser(NamedTuple):
    gender: str
    age: int
    id: int

input_collection = [
    CustomUser(gender='male', age=55, id=0),
    CustomUser(gender='female', age=25, id=1),
    CustomUser(gender='female', age=25, id=2),
    CustomUser(gender='female', age=33, id=3)
]

def RECORDREADER():
    return [(t.id, t) for t in input_collection]

def MAP(_, row: CustomUser):
    yield (row.gender, row.age)

def REDUCE_SUM(gender: str, ages: Iterator[int]):
    sum = 0
    for value in ages:
        sum += value
    yield (gender, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE_SUM)
output = list(output)
print("Сумма возрастов по гендеру:")
print(output)

Сумма возрастов по гендеру:
[('male', 55), ('female', 83)]


#

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [36]:
segment_size = 2 # Размер сегмента у вектора

class PartialResult(NamedTuple):
    row_index: int
    partial_sum: float

def RECORDREADER():
    for start_col in range(0, mat.shape[1], segment_size):
        end_col = start_col + segment_size
        v_segment = vec[start_col:end_col]
        m_strip = mat[:, start_col:end_col]
        yield (start_col, (start_col, m_strip, v_segment))

def MAP(_, data):
    start_col, m_strip, v_segment = data
    for i in range(m_strip.shape[0]):
        p_sum = np.dot(m_strip[i, :], v_segment)
        yield (i, p_sum)

def REDUCE(row_index: int, partial_sums: Iterator[float]):
    sum = 0
    for value in partial_sums:
        sum += value
    yield (row_index, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(2.671140012491868)),
 (1, np.float64(2.671140012491868)),
 (2, np.float64(2.671140012491868)),
 (3, np.float64(2.671140012491868)),
 (4, np.float64(2.671140012491868))]

## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$.





In [37]:
# MapReduce model
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

class MatrixElement(NamedTuple):
    matrix_id: str
    i: int
    j: int
    value: float

input_collection = [
    MatrixElement('M', 0, 0, 1),
    MatrixElement('M', 0, 1, 2),
    MatrixElement('M', 1, 0, 3), 
    MatrixElement('M', 1, 1, 4),
    MatrixElement('N', 0, 0, 5), 
    MatrixElement('N', 0, 1, 6),
    MatrixElement('N', 1, 0, 7), 
    MatrixElement('N', 1, 1, 8)
]

def MAP_JOIN(_, element: MatrixElement):
    if element.matrix_id == 'M':
        yield (element.j, ('M', element.i, element.value))
    else:
        yield (element.i, ('N', element.j, element.value))

def REDUCE_JOIN(j: int, values: Iterator):
    list_M = []
    list_N = []
    for v in values:
        if v[0] == 'M':
            list_M.append((v[1], v[2]))
        else:
            list_N.append((v[1], v[2]))
    for i, v in list_M:
        for k, w in list_N:
            yield ((i, k), v * w)

def MAP_SUM(ik, partial_mul):
    yield (ik, partial_mul)

def REDUCE_SUM(ik, partial_muls: Iterator[float]):
    sum = 0
    for value in partial_muls:
        sum += value
    yield (ik, sum)

def RECORDREADER_JOIN():
    return [(None, x) for x in input_collection]

join_output = MapReduce(RECORDREADER_JOIN, MAP_JOIN, REDUCE_JOIN)

def RECORDREADER_SUM():
    return list(join_output)

output = MapReduce(RECORDREADER_SUM, MAP_SUM, REDUCE_SUM)
output = list(output)
output

[((0, 0), 19), ((0, 1), 22), ((1, 0), 43), ((1, 1), 50)]

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [38]:
import numpy as np
I = 2
J = 3
K = 4*10
small_mat = np.random.rand(I,J) # it is legal to access this from RECORDREADER, MAP, REDUCE
big_mat = np.random.rand(J,K)

def RECORDREADER():
  for j in range(big_mat.shape[0]):
    for k in range(big_mat.shape[1]):
      yield ((j,k), big_mat[j,k])

def MAP(k1, v1):
    (j, k) = k1
    w = v1
    for i in range(small_mat.shape[0]):
        partial_product = small_mat[i, j] * w
        yield ((i, k), partial_product)

def REDUCE(key, values: Iterator[float]):
    (i, k) = key
    sum = 0
    for value in values:
        sum += value
    yield ((i, k), sum)

Проверьте своё решение

In [39]:
# CHECK THE SOLUTION
reference_solution = np.matmul(small_mat, big_mat)
solution = MapReduce(RECORDREADER, MAP, REDUCE)

def asmatrix(reduce_output):
  reduce_output = list(reduce_output)
  I = max(i for ((i,k), vw) in reduce_output)+1
  K = max(k for ((i,k), vw) in reduce_output)+1
  mat = np.empty(shape=(I,K))
  for ((i,k), vw) in reduce_output:
    mat[i,k] = vw
  return mat

np.allclose(reference_solution, asmatrix(solution)) # should return true

True

In [40]:
reduce_output = list(MapReduce(RECORDREADER, MAP, REDUCE))
max(i for ((i,k), vw) in reduce_output)

1

Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [41]:
M = [((0, 0), 1), ((0, 1), 2), ((1, 0), 3), ((1, 1), 4)]
N = [((0, 0), 5), ((0, 1), 6), ((1, 0), 7), ((1, 1), 8)]

def RECORDREADER_JOIN():
    for (i, j), v in M:
        yield (None, MatrixElement('M', i, j, v))
    for (j, k), v in N:
        yield (None, MatrixElement('N', j, k, v))

def MAP_JOIN(_, element: MatrixElement):
    if element.matrix_id == 'M':
        yield (element.j, ('M', element.i, element.value))
    else:
        yield (element.i, ('N', element.j, element.value))

def REDUCE_JOIN(j: int, values: Iterator):
    list_M = []
    list_N = []
    for v in values:
        if v[0] == 'M':
            list_M.append((v[1], v[2]))
        else:
            list_N.append((v[1], v[2]))
    for i, v in list_M:
        for k, w in list_N:
            yield ((i, k), v * w)

def MAP_SUM(ik, partial_mul):
    yield (ik, partial_mul)

def REDUCE_SUM(ik, partial_muls: Iterator[float]):
    sum = 0
    for value in partial_muls:
        sum += value
    yield (ik, sum)

join_output = MapReduce(RECORDREADER_JOIN, MAP_JOIN, REDUCE_JOIN)

def RECORDREADER_SUM():
    return list(join_output)

output = MapReduce(RECORDREADER_SUM, MAP_SUM, REDUCE_SUM)
output = list(output)
output

[((0, 0), 19), ((0, 1), 22), ((1, 0), 43), ((1, 1), 50)]

Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER.

In [42]:
M = [((0, 0), 1), ((0, 1), 2), ((1, 0), 3), ((1, 1), 4)]
N = [((0, 0), 5), ((0, 1), 6), ((1, 0), 7), ((1, 1), 8)]

reducers = 2

def INPUTFORMAT_JOIN():
    def RECORDREADER_M():
        for (i, j), v in M:
            yield (None, MatrixElement('M', i, j, v))
    def RECORDREADER_N():
        for (j, k), v in N:
            yield (None, MatrixElement('N', j, k, v))
    yield RECORDREADER_M()
    yield RECORDREADER_N()

def MAP_JOIN(_, element: MatrixElement):
    if element.matrix_id == 'M':
        yield (element.j, ('M', element.i, element.value))
    else:
        yield (element.i, ('N', element.j, element.value))

def REDUCE_JOIN(j: int, values: Iterator):
    list_M, list_N = [], []
    for v in values:
        if v[0] == 'M': list_M.append((v[1], v[2]))
        else: list_N.append((v[1], v[2]))
    for i, v in list_M:
        for k, w in list_N:
            yield ((i, k), v * w)

def MAP_SUM(ik, partial_mul):
    yield (ik, partial_mul)

def REDUCE_SUM(ik, partial_muls: Iterator[float]):
    sum = 0
    for value in partial_muls:
        sum += value
    yield (ik, sum)

def PARTITIONER(obj):
    global reducers
    return hash(obj) % reducers

join_output = MapReduceDistributed(INPUTFORMAT_JOIN, MAP_JOIN, REDUCE_JOIN, PARTITIONER)

join_step_data = []
for pid, partition in join_output:
    join_step_data.extend(list(partition))

def INPUTFORMAT_SUM():
    def RECORDREADER_SUM():
        for ik, val in join_step_data:
            yield (ik, val)
    yield RECORDREADER_SUM()

output_partitioned = MapReduceDistributed(INPUTFORMAT_SUM, MAP_SUM, REDUCE_SUM, PARTITIONER)

output = []
for pid, partition in output_partitioned:
    output.extend(list(partition))
sorted(output)

8 key-value pairs were sent over a network.
8 key-value pairs were sent over a network.


[((0, 0), 19), ((0, 1), 22), ((1, 0), 43), ((1, 1), 50)]

Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [43]:
reducers = 2

def INPUTFORMAT_JOIN():
    def RECORDREADER_M_part1():
        yield (None, MatrixElement('M', 0, 0, 1))
        yield (None, MatrixElement('M', 1, 1, 4))
        
    def RECORDREADER_M_part2():
        yield (None, MatrixElement('M', 0, 1, 2))
        yield (None, MatrixElement('M', 1, 0, 3))

    def RECORDREADER_N_part1():
        yield (None, MatrixElement('N', 0, 1, 6))
        yield (None, MatrixElement('N', 1, 1, 8))

    def RECORDREADER_N_part2():
        yield (None, MatrixElement('N', 0, 0, 5))
        yield (None, MatrixElement('N', 1, 0, 7))

    yield RECORDREADER_M_part1()
    yield RECORDREADER_M_part2()
    yield RECORDREADER_N_part1()
    yield RECORDREADER_N_part2()

def MAP_JOIN(_, element: MatrixElement):
    if element.matrix_id == 'M':
        yield (element.j, ('M', element.i, element.value))
    else:
        yield (element.i, ('N', element.j, element.value))

def REDUCE_JOIN(j: int, values: Iterator):
    list_M, list_N = [], []
    for v in values:
        if v[0] == 'M': list_M.append((v[1], v[2]))
        else: list_N.append((v[1], v[2]))
    for i, v in list_M:
        for k, w in list_N:
            yield ((i, k), v * w)

def MAP_SUM(ik, partial_mul):
    yield (ik, partial_mul)

def REDUCE_SUM(ik, partial_muls: Iterator[float]):
    sum = 0
    for value in partial_muls:
        sum += value
    yield (ik, sum)

def PARTITIONER(obj):
    return hash(obj) % reducers

join_output = MapReduceDistributed(INPUTFORMAT_JOIN, MAP_JOIN, REDUCE_JOIN, PARTITIONER)

join_step_data = []
for pid, partition in join_output:
    join_step_data.extend(list(partition))

def INPUTFORMAT_SUM():
    def RECORDREADER_SUM():
        for ik, val in join_step_data:
            yield (ik, val)
    yield RECORDREADER_SUM()

output_partitioned = MapReduceDistributed(INPUTFORMAT_SUM, MAP_SUM, REDUCE_SUM, PARTITIONER)

output = []
for pid, partition in output_partitioned:
    output.extend(list(partition))

sorted(output)

8 key-value pairs were sent over a network.
8 key-value pairs were sent over a network.


[((0, 0), 19), ((0, 1), 22), ((1, 0), 43), ((1, 1), 50)]

Если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы, то решение будет правильно работать при условии, что каждый i,j элемент сгенерирован единожды и каждый ненулевой элемент сгенерирован каким-либо RECORDREADER-ом